# Fraud Detection Model Predictions

This notebook loads a trained Spark MLlib Logistic Regression model and generates predictions on new fraud detection data from ClickHouse.

## Overview
- Load trained fraud detection model
- Fetch new data from ClickHouse 
- Apply same preprocessing pipeline as training
- Generate predictions and probability scores
- Analyze and visualize prediction results

## 1. Import Required Libraries

Import all necessary libraries for model loading, data handling, and predictions.

In [1]:
# Import required libraries
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder
from pyspark.ml.classification import LogisticRegressionModel
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.sql.functions import col, when
import pyspark.sql.functions as F
from clickhouse_driver import Client
import configparser
from pathlib import Path
import logging
import os
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure logging
log_filename = f"model_predictions_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
log_path = os.path.join('/root/research-dir/dev/jazzcash-fraud-detection/', log_filename)

# Create logger
logger = logging.getLogger('model_predictions')
logger.setLevel(logging.DEBUG)

# Create formatter
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')

# Create file handler
file_handler = logging.FileHandler(log_path)
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(formatter)

# Create console handler
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)
console_handler.setFormatter(formatter)

# Add handlers to logger
logger.addHandler(file_handler)
logger.addHandler(console_handler)

logger.info("✅ Libraries imported successfully")
logger.info(f"📝 Logging to file: {log_path}")

2025-11-05 16:12:49,994 - model_predictions - INFO - ✅ Libraries imported successfully
2025-11-05 16:12:49,995 - model_predictions - INFO - 📝 Logging to file: /root/research-dir/dev/jazzcash-fraud-detection/model_predictions_20251105_161249.log


In [2]:
# Initialize Spark Session
jar_files = [
    "/root/research-dir/dev/jazzcash-fraud-detection/utils/clickhouse-jdbc-0.9.2-all-dependencies.jar"
]

CLICKHOUSE_CONFIG = {
    'host': 'localhost',
    'port': 9000,  
    'database': 'public',
    'user': 'default',
    'password': 'DfsTeChB1'
}

url = f"jdbc:ch://{CLICKHOUSE_CONFIG['host']}:8123/{CLICKHOUSE_CONFIG['database']}"
user = CLICKHOUSE_CONFIG['user'] 
password = CLICKHOUSE_CONFIG['password']
driver = "com.clickhouse.jdbc.ClickHouseDriver"

try:
    spark.stop()
    logger.info("🔄 Stopped existing Spark session")
except:
    pass

spark = SparkSession.builder \
    .appName("fraud_predictions") \
    .master("spark://dfs-ai-app2:7077") \
    .config("spark.jars", ",".join(jar_files)) \
    .config("spark.executor.memory", "100g") \
    .config("spark.executor.memoryOverhead", "5g") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.cores", "32") \
    .config("spark.executor.instances", "2") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.default.parallelism", "96") \
    .getOrCreate()

logger.info("✅ Spark session initialized successfully")

25/11/05 16:12:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
2025-11-05 16:12:57,790 - model_predictions - INFO - ✅ Spark session initialized successfully


## 2. Load the Trained Model

Load the pre-trained Logistic Regression model from the saved location.

In [3]:
# Load the trained model
model_path = "/root/research-dir/dev/jazzcash-fraud-detection/scripts/spark_lr_model_june"

try:
    logger.info(f"📥 Loading trained model from: {model_path}")
    lr_model = LogisticRegressionModel.load(model_path)
    logger.info("✅ Model loaded successfully!")
    
    # Display model information
    logger.info("📊 Model Information:")
    logger.info(f"   • Number of features: {len(lr_model.coefficients)}")
    logger.info(f"   • Intercept: {lr_model.intercept:.6f}")
    logger.info(f"   • Decision threshold: {lr_model.getThreshold()}")
    
except Exception as e:
    logger.error(f"❌ Error loading model: {str(e)}")
    raise

2025-11-05 16:12:57,802 - model_predictions - INFO - 📥 Loading trained model from: /root/research-dir/dev/jazzcash-fraud-detection/scripts/spark_lr_model_june
2025-11-05 16:13:02,699 - model_predictions - INFO - ✅ Model loaded successfully!
2025-11-05 16:13:02,700 - model_predictions - INFO - 📊 Model Information:
2025-11-05 16:13:02,727 - model_predictions - INFO -    • Number of features: 290
2025-11-05 16:13:02,728 - model_predictions - INFO -    • Intercept: -10.376682
2025-11-05 16:13:02,729 - model_predictions - INFO -    • Decision threshold: 0.5


## 3. Load and Prepare Test Data

Load new data from ClickHouse and apply the same preprocessing pipeline as used during training.

In [ ]:
# Define the same feature columns used in training
selected_cols = [
    'cutoff_date',
    'fraud_flag',
    'trx_channel',
    'trx_type',
    'start_balance',
    'trx_amt',
    'mbar_registered_channel',
    'mbar_a_c_status',
    'mbar_a_c_level',
    'mbar_account_type_name',
    'hour_of_day',
    'day_of_week',
    'is_weekend',
    'is_night',
    'is_business_hours',
    'is_unusual_hour',
    'night_weekend_combo',
    'start_balance_log',
    'txn_txns_3d',
    'txn_total_amount_3d',
    'txn_avg_amount_3d',
    'txn_max_amount_3d',
    'txn_min_amount_3d',
    'txn_unique_recipients_3d',
    'txn_unique_channels_3d',
    'txn_unique_types_3d',
    'txn_is_high_activity_3d',
    'txn_multi_channel_recent',
    'txn_amount_deviation_from_avg',
    'txn_night_txns_3d',
    'txn_weekend_txns_3d',
    'channel_new_jc_app',
    'channel_ussd',
    'channel_ussd_api',
    'channel_payment_gateway',
    'channel_mobile_app',
    'type_transfer_c2c',
    'type_transfer_c2b',
    'type_bill_payment',
    'type_mobile_load',
    'user_total_txns_3d',
    'user_total_amount_3d',
    'user_avg_amount_3d',
    'user_median_amount_3d',
    'user_max_amount_3d',
    'user_min_amount_3d',
    'user_unique_recipients_3d',
    'user_unique_channels_3d',
    'user_unique_types_3d',
    'user_total_txns_7d',
    'user_total_amount_7d',
    'user_avg_amount_7d',
    'user_median_amount_7d',
    'user_max_amount_7d',
    'user_min_amount_7d',
    'user_unique_recipients_7d',
    'user_unique_channels_7d',
    'user_unique_types_7d',
    'user_most_used_channel_7d',
    'user_last_used_channel',
    'user_channel_diversity_score_7d',
    'user_most_used_type_7d',
    'user_last_used_type',
    'user_type_diversity_score_7d',
    'user_night_txns_7d',
    'user_weekend_txns_7d',
    'user_peak_hour_txns_7d',
    'user_off_peak_hour_txns_7d',
    'user_avg_start_balance_7d',
    'user_avg_end_balance_7d',
    'user_min_balance_7d',
    'user_max_balance_7d',
    'user_balance_volatility_7d',
    'user_avg_amount_per_recipient_7d',
    'user_max_amount_to_single_recipient_7d',
    'user_recipient_concentration_ratio_7d',
    'user_avg_time_between_txns_7d',
    'user_txn_frequency_score_7d',
    'user_days_since_last_txn'
]

# Load test data (using July 2025 data for predictions)
start_date = '2025-06-01'
end_date = '2025-06-30'
num_partitions = 30

query = f"""
    SELECT {', '.join(selected_cols)}
    FROM stixor_fraud_features_distributed
    WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
        AND mbar_account_type_name = 'Customer Account'
"""

subquery = f"""
(
    {query}
) AS fraud_data
"""

logger.info(f"🔍 Loading test data from {start_date} to {end_date}...")

start_time = datetime.now()
try:
    df_test = (spark.read
        .format('jdbc')
        .option('driver', driver)
        .option('url', url)
        .option('user', user)
        .option('password', password)
        .option('dbtable', subquery)
        .option('fetchsize', '100000')
        .option("partitionColumn", "cutoff_date")
        .option('lowerBound', start_date)
        .option('upperBound', end_date)
        .option('numPartitions', str(num_partitions))
        .load())
    
    # Cache the DataFrame
    df_test.cache()
    
    # Count rows
    total_rows = df_test.count()
    
    end_time = datetime.now()
    duration = (end_time - start_time).total_seconds()
    
    logger.info("✅ TEST DATA LOADED SUCCESSFULLY!")
    logger.info(f"   • Total rows: {total_rows:,}")
    logger.info(f"   • Loading time: {duration:.2f} seconds")
    
except Exception as e:
    logger.error(f"❌ Error loading test data: {str(e)}")
    raise

2025-11-05 16:17:33,573 - model_predictions - INFO - 🔍 Loading test data from 2025-06-01 to 2025-06-30...
25/11/05 16:17:33 WARN JDBCRelation: The number of partitions is reduced because the specified number of partitions is less than the difference between upper bound and lower bound. Updated number of partitions: 29; Input number of partitions: 30; Lower bound: '2025-06-01'; Upper bound: '2025-06-30'.


In [7]:
# Apply the same preprocessing pipeline as training
logger.info("🔧 Applying preprocessing pipeline...")

TARGET_COLUMN = 'fraud_flag'

# Check target distribution in test data
logger.info(f"📈 Test Data Target Distribution:")
df_test.groupBy(TARGET_COLUMN).count().show()

# Get feature columns (exclude non-predictive columns)
excluded_columns = [
    TARGET_COLUMN,
    'processed_date',
    'ac_from',
    'ac_to',
    'msisdn_from',
    'msisdn_to',
    'transaction_uuid'
]

all_feature_cols = [col_name for col_name in df_test.columns if col_name not in excluded_columns]

# Identify string (categorical) columns
string_cols = []
for field in df_test.schema.fields:
    if field.name in all_feature_cols and field.dataType.typeName() == 'string':
        string_cols.append(field.name)

logger.info(f"🔤 String columns to encode: {string_cols}")

# Replace empty strings with 'UNKNOWN'
for col_name in string_cols:
    df_test = df_test.withColumn(col_name, when((F.col(col_name) == "") | F.col(col_name).isNull(), "UNKNOWN").otherwise(F.col(col_name)))

# Apply string indexing and one-hot encoding
indexers = [
    StringIndexer(inputCol=col, outputCol=f"{col}_idx", handleInvalid="keep") for col in string_cols
]
encoders = [
    OneHotEncoder(inputCol=f"{col}_idx", outputCol=f"{col}_ohe", handleInvalid="keep") for col in string_cols
]

# Apply categorical pipeline
cat_pipeline = Pipeline(stages=indexers + encoders)
df_cat_test = cat_pipeline.fit(df_test).transform(df_test)

# Update feature columns to use OHE versions
modeling_features = [
    f"{col}_ohe" if col in string_cols else col for col in all_feature_cols
]

# Remove columns not present in df_cat_test
modeling_features = [col for col in modeling_features if col in df_cat_test.columns]

# Remove the first feature (cutoff_date) as done in training
modeling_features = modeling_features[1:]

logger.info(f"🧮 Final modeling features: {len(modeling_features)} features")

df_clean_test = df_cat_test

2025-11-05 16:14:07,509 - model_predictions - INFO - 🔧 Applying preprocessing pipeline...
2025-11-05 16:14:07,511 - model_predictions - INFO - 📈 Test Data Target Distribution:
2025-11-05 16:14:08,498 - model_predictions - INFO - 🔤 String columns to encode: ['trx_channel', 'trx_type', 'mbar_registered_channel', 'mbar_a_c_status', 'mbar_a_c_level', 'mbar_account_type_name', 'user_most_used_channel_7d', 'user_last_used_channel', 'user_most_used_type_7d', 'user_last_used_type']


+----------+-----+
|fraud_flag|count|
+----------+-----+
|         0| 5893|
+----------+-----+



2025-11-05 16:14:12,404 - model_predictions - INFO - 🧮 Final modeling features: 77 features


In [11]:
# FEATURE MISMATCH FIX: Load training data to get consistent feature encoding
logger.info("🔧 FIXING FEATURE MISMATCH - Loading training data for consistent encoding...")

# Load a small sample of training data to get the same categorical encodings
train_start_date = '2025-06-01'
train_end_date = '2025-06-30'  # Just 2 days to get the categorical mappings

train_query = f"""
    SELECT {', '.join(selected_cols)}
    FROM stixor_fraud_features_distributed
    WHERE cutoff_date BETWEEN '{train_start_date}' AND '{train_end_date}'
        AND mbar_account_type_name = 'Customer Account'
    LIMIT 1000
"""

train_subquery = f"""
(
    {train_query}
) AS train_data
"""

# Load small training sample
df_train_sample = (spark.read
    .format('jdbc')
    .option('driver', driver)
    .option('url', url)
    .option('user', user)
    .option('password', password)
    .option('dbtable', train_subquery)
    .load())

logger.info(f"✅ Loaded training sample: {df_train_sample.count()} rows")

# Combine training sample with test data for consistent encoding
logger.info("🔧 Combining training and test data for consistent categorical encoding...")

# Add a flag to distinguish training vs test data
df_train_sample = df_train_sample.withColumn("data_type", F.lit("train"))
df_test_flagged = df_test.withColumn("data_type", F.lit("test"))

# Union the datasets
df_combined = df_train_sample.union(df_test_flagged)

logger.info(f"✅ Combined dataset created: {df_combined.count()} rows")

# Now apply preprocessing to the combined dataset
logger.info("🔧 Applying preprocessing to combined dataset...")

# Replace empty strings with 'UNKNOWN' for all data
for col_name in string_cols:
    df_combined = df_combined.withColumn(col_name, when((F.col(col_name) == "") | F.col(col_name).isNull(), "UNKNOWN").otherwise(F.col(col_name)))

# Apply categorical encoding to combined dataset
cat_pipeline_combined = Pipeline(stages=indexers + encoders)
cat_pipeline_fitted = cat_pipeline_combined.fit(df_combined)
df_combined_encoded = cat_pipeline_fitted.transform(df_combined)

# Split back into test data only
df_test_fixed = df_combined_encoded.filter(F.col("data_type") == "test").drop("data_type")

logger.info("✅ Test data with consistent encoding created")

# Update feature columns with the new encoding
modeling_features_fixed = [
    f"{col}_ohe" if col in string_cols else col for col in all_feature_cols
]
modeling_features_fixed = [col for col in modeling_features_fixed if col in df_test_fixed.columns]
modeling_features_fixed = modeling_features_fixed[1:]  # Remove cutoff_date

logger.info(f"🧮 Fixed modeling features: {len(modeling_features_fixed)} features")

# Update the clean test dataframe
df_clean_test = df_test_fixed

2025-11-05 16:16:21,870 - model_predictions - INFO - 🔧 FIXING FEATURE MISMATCH - Loading training data for consistent encoding...
2025-11-05 16:16:22,135 - model_predictions - INFO - ✅ Loaded training sample: 1000 rows
2025-11-05 16:16:22,136 - model_predictions - INFO - 🔧 Combining training and test data for consistent categorical encoding...
2025-11-05 16:16:22,281 - model_predictions - INFO - ✅ Combined dataset created: 6893 rows
2025-11-05 16:16:22,282 - model_predictions - INFO - 🔧 Applying preprocessing to combined dataset...
2025-11-05 16:16:25,133 - model_predictions - INFO - ✅ Test data with consistent encoding created
2025-11-05 16:16:25,178 - model_predictions - INFO - 🧮 Fixed modeling features: 77 features


In [12]:
# Create feature vectors and scale features (same as training)
logger.info("🔧 Creating feature vectors...")

# Assemble features
assembler = VectorAssembler(
    inputCols=modeling_features,
    outputCol="raw_features",
    handleInvalid="skip"
)

df_assembled_test = assembler.transform(df_clean_test)

# Scale features
scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withStd=True,
    withMean=True
)

# Fit scaler on test data (in production, you'd use the same scaler from training)
scaler_model_test = scaler.fit(df_assembled_test)
df_scaled_test = scaler_model_test.transform(df_assembled_test)

# Prepare final test dataset
df_final_test = df_scaled_test.select("features", col(TARGET_COLUMN).alias("label"))

logger.info("✅ Test data preprocessing completed!")
logger.info(f"   • Features: Vector of {len(modeling_features)} elements")
logger.info(f"   • Test records: {df_final_test.count():,}")

2025-11-05 16:16:30,226 - model_predictions - INFO - 🔧 Creating feature vectors...
2025-11-05 16:16:30,900 - model_predictions - INFO - ✅ Test data preprocessing completed!
2025-11-05 16:16:30,901 - model_predictions - INFO -    • Features: Vector of 77 elements
2025-11-05 16:16:31,400 - model_predictions - INFO -    • Test records: 4,126


## 4. Generate Predictions

Use the loaded model to make predictions on the test data.

In [ ]:
# Generate predictions using the trained model
logger.info("🎯 Generating predictions...")

start_time = datetime.now()

# Make predictions
predictions = lr_model.transform(df_final_test)

# Cache predictions for multiple operations
predictions.cache()

# Trigger evaluation to cache results
prediction_count = predictions.count()

end_time = datetime.now()
prediction_time = (end_time - start_time).total_seconds()

logger.info("✅ Predictions generated successfully!")
logger.info(f"   • Total predictions: {prediction_count:,}")
logger.info(f"   • Prediction time: {prediction_time:.2f} seconds")

# Show sample predictions
logger.info("📋 Sample Predictions:")
predictions.select("label", "prediction", "probability").show(10, truncate=False)

2025-11-05 16:16:47,703 - model_predictions - INFO - 🎯 Generating predictions...
25/11/05 16:16:48 WARN TaskSetManager: Lost task 20.0 in stage 219.0 (TID 1364) (10.205.161.118 executor 1): org.apache.spark.SparkException: [FAILED_EXECUTE_UDF] User defined function (`ProbabilisticClassificationModel$$Lambda/0x00007f44792bf280`: (struct<type:tinyint,size:int,indices:array<int>,values:array<double>>) => struct<type:tinyint,size:int,indices:array<int>,values:array<double>>) failed due to: java.lang.IllegalArgumentException: requirement failed: BLAS.dot(x: Vector, y:Vector) was given Vectors with non-matching sizes: x.size = 197, y.size = 290. SQLSTATE: 39000
	at org.apache.spark.sql.errors.QueryExecutionErrors$.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala:195)
	at org.apache.spark.sql.errors.QueryExecutionErrors.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage

Py4JJavaError: An error occurred while calling o4768.count.
: org.apache.spark.SparkException: [FAILED_EXECUTE_UDF] User defined function (`ProbabilisticClassificationModel$$Lambda/0x00007f44792bf280`: (struct<type:tinyint,size:int,indices:array<int>,values:array<double>>) => struct<type:tinyint,size:int,indices:array<int>,values:array<double>>) failed due to: java.lang.IllegalArgumentException: requirement failed: BLAS.dot(x: Vector, y:Vector) was given Vectors with non-matching sizes: x.size = 197, y.size = 290. SQLSTATE: 39000
	at org.apache.spark.sql.errors.QueryExecutionErrors$.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala:195)
	at org.apache.spark.sql.errors.QueryExecutionErrors.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage3.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at or

25/11/05 16:16:48 WARN TaskSetManager: Lost task 0.0 in stage 219.0 (TID 1374) (10.205.161.118 executor 1): TaskKilled (Stage cancelled: [FAILED_EXECUTE_UDF] User defined function (`ProbabilisticClassificationModel$$Lambda/0x00007f44792bf280`: (struct<type:tinyint,size:int,indices:array<int>,values:array<double>>) => struct<type:tinyint,size:int,indices:array<int>,values:array<double>>) failed due to: java.lang.IllegalArgumentException: requirement failed: BLAS.dot(x: Vector, y:Vector) was given Vectors with non-matching sizes: x.size = 197, y.size = 290. SQLSTATE: 39000)


## 5. Display and Analyze Results

Display prediction results and perform analysis of the model output including metrics and visualizations.

In [ ]:
# Evaluate model performance on test data
logger.info("📊 Evaluating model performance...")

# Binary Classification Metrics
binary_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc_roc = binary_evaluator.evaluate(predictions)

# Area Under PR Curve
binary_evaluator_pr = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)

auc_pr = binary_evaluator_pr.evaluate(predictions)

# Multiclass metrics
multi_evaluator_accuracy = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

multi_evaluator_precision = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

multi_evaluator_recall = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)

multi_evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

accuracy = multi_evaluator_accuracy.evaluate(predictions)
precision = multi_evaluator_precision.evaluate(predictions)
recall = multi_evaluator_recall.evaluate(predictions)
f1_score = multi_evaluator_f1.evaluate(predictions)

logger.info("✅ Model Performance Metrics:")
logger.info(f"   • AUC-ROC: {auc_roc:.4f}")
logger.info(f"   • AUC-PR: {auc_pr:.4f}")
logger.info(f"   • Accuracy: {accuracy:.4f}")
logger.info(f"   • Precision: {precision:.4f}")
logger.info(f"   • Recall: {recall:.4f}")
logger.info(f"   • F1-Score: {f1_score:.4f}")

In [ ]:
# Confusion Matrix Analysis
logger.info("🔍 Creating Confusion Matrix...")

# Create confusion matrix
confusion_matrix = predictions.groupBy("label", "prediction").count().collect()

# Convert to a more readable format
cm_data = {}
for row in confusion_matrix:
    actual = int(row['label'])
    predicted = int(row['prediction'])
    count = row['count']
    cm_data[(actual, predicted)] = count

# Extract values for 2x2 matrix
tn = cm_data.get((0, 0), 0)  # True Negatives
fp = cm_data.get((0, 1), 0)  # False Positives
fn = cm_data.get((1, 0), 0)  # False Negatives
tp = cm_data.get((1, 1), 0)  # True Positives

logger.info("📈 Confusion Matrix:")
logger.info(f"   • True Negatives (TN): {tn:,}")
logger.info(f"   • False Positives (FP): {fp:,}")
logger.info(f"   • False Negatives (FN): {fn:,}")
logger.info(f"   • True Positives (TP): {tp:,}")

# Calculate additional metrics
if tp + fp > 0:
    precision_fraud = tp / (tp + fp)
else:
    precision_fraud = 0

if tp + fn > 0:
    recall_fraud = tp / (tp + fn)
else:
    recall_fraud = 0

if precision_fraud + recall_fraud > 0:
    f1_fraud = 2 * (precision_fraud * recall_fraud) / (precision_fraud + recall_fraud)
else:
    f1_fraud = 0

logger.info("🎯 Fraud Detection Specific Metrics:")
logger.info(f"   • Fraud Precision: {precision_fraud:.4f}")
logger.info(f"   • Fraud Recall: {recall_fraud:.4f}")
logger.info(f"   • Fraud F1-Score: {f1_fraud:.4f}")
logger.info(f"   • False Positive Rate: {fp/(fp+tn):.4f}" if (fp+tn) > 0 else "   • False Positive Rate: N/A")

In [ ]:
# Prediction Distribution Analysis
logger.info("📊 Analyzing prediction distributions...")

# Get prediction distribution
pred_distribution = predictions.groupBy("prediction").count().collect()

logger.info("🔢 Prediction Distribution:")
for row in pred_distribution:
    pred_class = int(row['prediction'])
    count = row['count']
    percentage = (count / prediction_count) * 100
    class_name = "Legitimate" if pred_class == 0 else "Fraud"
    logger.info(f"   • {class_name}: {count:,} ({percentage:.2f}%)")

# Analyze probability distributions
# Extract probability for fraud class (class 1)
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType
import numpy as np

def extract_prob_fraud(probability_vector):
    """Extract probability for fraud class (index 1)"""
    if probability_vector is not None and len(probability_vector) > 1:
        return float(probability_vector[1])
    return 0.0

extract_prob_udf = udf(extract_prob_fraud, DoubleType())

# Add fraud probability column
predictions_with_prob = predictions.withColumn("fraud_probability", extract_prob_udf("probability"))

# Show high-risk predictions (fraud probability > 0.8)
logger.info("⚠️  High Risk Predictions (Fraud Probability > 0.8):")
high_risk = predictions_with_prob.filter(predictions_with_prob.fraud_probability > 0.8)
high_risk_count = high_risk.count()
logger.info(f"   • Count: {high_risk_count:,}")

if high_risk_count > 0:
    high_risk.select("label", "prediction", "fraud_probability").show(10)

In [ ]:
# Convert results to Pandas for visualization
logger.info("📈 Creating visualizations...")

# Sample predictions for visualization (to avoid memory issues)
sample_size = min(10000, prediction_count)
predictions_sample = predictions_with_prob.sample(False, sample_size/prediction_count, seed=42)

# Convert to Pandas
pred_pandas = predictions_sample.select("label", "prediction", "fraud_probability").toPandas()

# Create visualizations
plt.figure(figsize=(15, 10))

# 1. Confusion Matrix Heatmap
plt.subplot(2, 3, 1)
cm_matrix = np.array([[tn, fp], [fn, tp]])
sns.heatmap(cm_matrix, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Predicted Legitimate', 'Predicted Fraud'],
            yticklabels=['Actual Legitimate', 'Actual Fraud'])
plt.title('Confusion Matrix')

# 2. Fraud Probability Distribution
plt.subplot(2, 3, 2)
plt.hist(pred_pandas['fraud_probability'], bins=50, alpha=0.7, color='skyblue', edgecolor='black')
plt.xlabel('Fraud Probability')
plt.ylabel('Frequency')
plt.title('Distribution of Fraud Probabilities')

# 3. Fraud Probability by Actual Label
plt.subplot(2, 3, 3)
legitimate = pred_pandas[pred_pandas['label'] == 0]['fraud_probability']
fraud = pred_pandas[pred_pandas['label'] == 1]['fraud_probability']

plt.hist(legitimate, bins=30, alpha=0.7, label='Legitimate', color='green')
plt.hist(fraud, bins=30, alpha=0.7, label='Fraud', color='red')
plt.xlabel('Fraud Probability')
plt.ylabel('Frequency')
plt.title('Fraud Probability by Actual Label')
plt.legend()

# 4. Prediction Distribution Pie Chart
plt.subplot(2, 3, 4)
pred_counts = pred_pandas['prediction'].value_counts()
plt.pie(pred_counts.values, labels=['Legitimate', 'Fraud'], autopct='%1.1f%%', 
        colors=['lightgreen', 'lightcoral'])
plt.title('Prediction Distribution')

# 5. Performance Metrics Bar Chart
plt.subplot(2, 3, 5)
metrics = ['AUC-ROC', 'AUC-PR', 'Accuracy', 'Precision', 'Recall', 'F1-Score']
values = [auc_roc, auc_pr, accuracy, precision, recall, f1_score]
bars = plt.bar(metrics, values, color=['skyblue', 'lightgreen', 'orange', 'purple', 'pink', 'yellow'])
plt.ylim(0, 1)
plt.title('Model Performance Metrics')
plt.xticks(rotation=45)

# Add value labels on bars
for bar, value in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{value:.3f}', ha='center', va='bottom')

# 6. ROC Curve approximation (simplified)
plt.subplot(2, 3, 6)
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.plot([0, recall_fraud, 1], [0, 1-precision_fraud, 1], 'b-', label=f'Model (AUC = {auc_roc:.3f})')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve (Simplified)')
plt.legend()

plt.tight_layout()
plt.savefig('/root/research-dir/dev/jazzcash-fraud-detection/model_predictions_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

logger.info("✅ Visualizations created and saved!")
logger.info("📁 Saved to: /root/research-dir/dev/jazzcash-fraud-detection/model_predictions_analysis.png")

In [ ]:
# Save prediction results
logger.info("💾 Saving prediction results...")

# Save detailed predictions to CSV
output_path = "/root/research-dir/dev/jazzcash-fraud-detection/fraud_predictions_july_2025.csv"

# Select relevant columns for output
output_df = predictions_with_prob.select(
    "label", 
    "prediction", 
    "fraud_probability"
).coalesce(1)  # Combine into single file

# Save to CSV
output_df.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(output_path)

logger.info(f"✅ Predictions saved to: {output_path}")

# Create summary report
summary_report = f"""
FRAUD DETECTION MODEL PREDICTIONS SUMMARY
=========================================

Data Period: July 2025
Model: Logistic Regression (trained on June 2025 data)
Total Predictions: {prediction_count:,}

PERFORMANCE METRICS:
- AUC-ROC: {auc_roc:.4f}
- AUC-PR: {auc_pr:.4f}
- Accuracy: {accuracy:.4f}
- Precision: {precision:.4f}
- Recall: {recall:.4f}
- F1-Score: {f1_score:.4f}

CONFUSION MATRIX:
- True Negatives: {tn:,}
- False Positives: {fp:,}
- False Negatives: {fn:,}
- True Positives: {tp:,}

FRAUD-SPECIFIC METRICS:
- Fraud Precision: {precision_fraud:.4f}
- Fraud Recall: {recall_fraud:.4f}
- Fraud F1-Score: {f1_fraud:.4f}
- False Positive Rate: {fp/(fp+tn):.4f if (fp+tn) > 0 else 'N/A'}

PREDICTION DISTRIBUTION:
"""

for row in pred_distribution:
    pred_class = int(row['prediction'])
    count = row['count']
    percentage = (count / prediction_count) * 100
    class_name = "Legitimate" if pred_class == 0 else "Fraud"
    summary_report += f"- {class_name}: {count:,} ({percentage:.2f}%)\n"

summary_report += f"""
HIGH-RISK PREDICTIONS (>80% fraud probability): {high_risk_count:,}

FILES GENERATED:
- Predictions CSV: {output_path}
- Analysis Charts: /root/research-dir/dev/jazzcash-fraud-detection/model_predictions_analysis.png
- Log File: {log_path}

Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
"""

# Save summary report
summary_path = "/root/research-dir/dev/jazzcash-fraud-detection/fraud_predictions_summary.txt"
with open(summary_path, 'w') as f:
    f.write(summary_report)

print(summary_report)
logger.info(f"📋 Summary report saved to: {summary_path}")
logger.info("🎉 FRAUD DETECTION PREDICTIONS COMPLETED SUCCESSFULLY!")

## Summary

This notebook successfully:

1. **Loaded the trained model** - Imported the pre-trained Logistic Regression model from the saved location
2. **Processed test data** - Applied the same preprocessing pipeline used during training to July 2025 data
3. **Generated predictions** - Used the model to predict fraud probabilities for new transactions
4. **Evaluated performance** - Calculated comprehensive metrics including AUC-ROC, precision, recall, and F1-score
5. **Created visualizations** - Generated charts showing confusion matrix, probability distributions, and performance metrics
6. **Saved results** - Exported predictions to CSV and created a detailed summary report

### Key Outputs:
- **Prediction CSV**: Contains all predictions with fraud probabilities
- **Analysis Charts**: Visual analysis of model performance
- **Summary Report**: Comprehensive performance summary
- **Log Files**: Detailed execution logs

### Next Steps:
- Review high-risk predictions for manual investigation
- Adjust decision threshold based on business requirements
- Monitor model performance over time
- Consider retraining if performance degrades